# 06 Forecasting Feature Engineering

Notebook nay tao modeling dataset, feature catalog, so sanh model, va luu model candidate. Moi feature daily exogenous deu duoc lag 1 ngay de tranh leakage.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display
from joblib import dump
from sklearn.base import clone

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'dataset').exists():
            return candidate
    raise FileNotFoundError('Could not locate project root from the current working directory.')

PROJECT_ROOT = find_project_root(Path.cwd()).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.competition_workflow import (
    build_feature_list,
    build_modeling_dataset,
    build_training_frame,
    ensure_stage_directories,
    evaluate_model_candidates,
    export_model_comparison_artifacts,
    export_modeling_artifacts,
    get_model_candidates,
    )

dirs = ensure_stage_directories(PROJECT_ROOT)
pd.set_option('display.max_columns', None)

In [2]:
modeling_dataset = build_modeling_dataset(PROJECT_ROOT)
feature_list = build_feature_list(modeling_dataset)
display(modeling_dataset.head())
display(feature_list.head(20))
modeling_dataset.isna().mean().sort_values(ascending=False).head(20)

,date,Revenue,COGS,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,daily_order_count,daily_qty_sold,daily_line_revenue,daily_active_promos,avg_discount_value,daily_returns,daily_refund_amount,daily_avg_rating,daily_avg_delay,month_key,monthly_fill_rate,monthly_stockout_days,monthly_sell_through_rate,dayofweek,month,quarter,year,dayofmonth,weekofyear,is_weekend,is_month_end,time_index,revenue_lag_1,revenue_lag_7,revenue_lag_14,revenue_lag_28,revenue_roll_mean_7,revenue_roll_std_7,revenue_roll_mean_14,revenue_roll_std_14,revenue_roll_mean_30,revenue_roll_std_30,cogs_lag_1,cogs_lag_7,cogs_lag_14,cogs_lag_28,cogs_roll_mean_7,cogs_roll_std_7,cogs_roll_mean_14,cogs_roll_std_14,cogs_roll_mean_30,cogs_roll_std_30
0,2012-07-04,5123547.94,3982991.19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2012-07,0.966075,402,0.243548,2,7,3,2012,4,27,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012-07-05,2751773.45,2150580.23,NaN,NaN,NaN,NaN,NaN,162.0,777.0,5123547.94,NaN,NaN,NaN,NaN,NaN,4.333333,2012-07,0.966075,402,0.243548,3,7,3,2012,5,27,0,0,1,5123547.94,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3982991.19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2012-07-06,3054029.42,2517632.84,NaN,NaN,NaN,NaN,NaN,97.0,428.0,2751773.45,NaN,NaN,NaN,NaN,NaN,4.444444,2012-07,0.966075,402,0.243548,4,7,3,2012,6,27,0,0,2,2751773.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2150580.23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2012-07-07,2667930.94,2108246.62,NaN,NaN,NaN,NaN,NaN,93.0,441.0,3054029.42,NaN,NaN,NaN,NaN,NaN,4.563830,2012-07,0.966075,402,0.243548,5,7,3,2012,7,27,1,0,3,3054029.42,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2517632.84,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2012-07-08,2360851.90,1808622.79,NaN,NaN,NaN,NaN,NaN,73.0,364.0,2667930.94,NaN,NaN,NaN,NaN,NaN,4.336957,2012-07,0.966075,402,0.243548,6,7,3,2012,8,27,1,0,4,2667930.94,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2108246.62,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,feature,group,description
0,sessions,web_traffic,Auto-generated feature for sessions.
1,unique_visitors,web_traffic,Auto-generated feature for unique visitors.
2,page_views,web_traffic,Auto-generated feature for page views.
3,bounce_rate,web_traffic,Auto-generated feature for bounce rate.
4,avg_session_duration_sec,web_traffic,Auto-generated feature for avg session duratio...
5,daily_order_count,daily_aggregate,Auto-generated feature for daily order count.
6,daily_qty_sold,daily_aggregate,Auto-generated feature for daily qty sold.
7,daily_line_revenue,daily_aggregate,Auto-generated feature for daily line revenue.
8,daily_active_promos,daily_aggregate,Auto-generated feature for daily active promos.
9,avg_discount_value,daily_aggregate,Auto-generated feature for avg discount value.


daily_active_promos         0.554918
avg_discount_value          0.554918
avg_session_duration_sec    0.047482
bounce_rate                 0.047482
sessions                    0.047482
unique_visitors             0.047482
page_views                  0.047482
revenue_roll_mean_30        0.007827
cogs_roll_std_30            0.007827
cogs_roll_mean_30           0.007827
revenue_roll_std_30         0.007827
daily_returns               0.007305
cogs_lag_28                 0.007305
revenue_lag_28              0.007305
daily_refund_amount         0.007305
cogs_lag_14                 0.003652
revenue_lag_14              0.003652
revenue_roll_mean_14        0.003652
cogs_roll_mean_14           0.003652
revenue_roll_std_14         0.003652
dtype: float64

In [3]:
model_scores = evaluate_model_candidates(modeling_dataset, target_col='Revenue', n_splits=5)
display(model_scores)

c:\Users\LENOVO\anaconda3\envs\datathon\lib\site-packages\sklearn\linear_model\_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=7.95345e-17): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\LENOVO\anaconda3\envs\datathon\lib\site-packages\sklearn\linear_model\_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=3.81109e-17): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\LENOVO\anaconda3\envs\datathon\lib\site-packages\sklearn\linear_model\_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=2.22059e-17): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\LENOVO\anaconda3\envs\datathon\lib\site-packages\sklearn\linear_model\_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=1.55314e-17): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\LENOVO\

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001459 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3182
[LightGBM] [Info] Number of data points in the train set: 283, number of used features: 44
[LightGBM] [Info] Start training from score 4682440.453401
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best g

,Model,MAE,RMSE,R2,Folds
1,XGBoost,735242.330320,1.051396e+06,0.754893,5
2,LightGBM,739215.923189,1.052064e+06,0.754427,5
0,Ridge,793608.426321,1.100910e+06,0.724909,5


In [4]:
candidate_models = get_model_candidates()
best_model_name = model_scores.iloc[0]['Model']
X, y = build_training_frame(modeling_dataset, target_col='Revenue')
best_model = clone(candidate_models[best_model_name])
best_model.fit(X, y)
model_path = dirs['models'] / 'model_candidate.pkl'
dump({
    'target': 'Revenue',
    'model_name': best_model_name,
    'feature_columns': X.columns.tolist(),
    'model': best_model,
}, model_path)
print(f'Saved model candidate to: {model_path.relative_to(PROJECT_ROOT)}')

Saved model candidate to: models\model_candidate.pkl


In [5]:
artifact_paths = {}
artifact_paths.update(export_modeling_artifacts(PROJECT_ROOT))
artifact_paths.update(export_model_comparison_artifacts(PROJECT_ROOT, target_col='Revenue'))
pd.DataFrame({
    'artifact': list(artifact_paths.keys()),
    'path': [str(path.relative_to(PROJECT_ROOT)) for path in artifact_paths.values()],
}).sort_values('artifact')

c:\Users\LENOVO\anaconda3\envs\datathon\lib\site-packages\sklearn\linear_model\_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=7.95345e-17): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\LENOVO\anaconda3\envs\datathon\lib\site-packages\sklearn\linear_model\_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=3.81109e-17): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\LENOVO\anaconda3\envs\datathon\lib\site-packages\sklearn\linear_model\_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=2.22059e-17): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\LENOVO\anaconda3\envs\datathon\lib\site-packages\sklearn\linear_model\_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=1.55314e-17): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\LENOVO\

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000822 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3182
[LightGBM] [Info] Number of data points in the train set: 283, number of used features: 44
[LightGBM] [Info] Start training from score 4682440.453401
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best g

,artifact,path
1,feature_list,artifacts\modeling\feature_list.csv
2,model_candidate,models\model_candidate.pkl
3,model_vs_baseline,artifacts\modeling\model_vs_baseline.csv
0,modeling_dataset,artifacts\modeling\modeling_dataset.parquet
